# 01 · Modelo en estrella para Power BI y Tableau

Construye las cuatro tablas de `data/modelo/` a partir de ficheros versionados:

- `dim_equipo.csv`, `dim_posicion.csv`, `dim_jugador.csv`
- `hechos_jugador.csv`: métricas por 90 minutos y percentiles por grupo de posición

El cruce de nombres entre fuentes no se hace aquí: se lee de `data/mapa_nombres.csv`, que genera y
documenta el cuaderno 00.

**Cobertura:** jugadores de campo con minutos en LaLiga 2025/26. Los porteros quedan fuera de esta versión.

In [1]:
# Instalamos las librerías del cuaderno; si ya están instaladas no cambia nada
%pip install -q pandas numpy unidecode

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importamos os para situarnos en la raíz del proyecto
import os
# Importamos numpy para los percentiles
import numpy as np
# Importamos pandas para trabajar con tablas
import pandas as pd
# Importamos unidecode para generar nombres de fichero sin acentos
from unidecode import unidecode

# Si el cuaderno se ejecuta desde notebooks/, subimos a la raíz para usar rutas relativas
if os.path.basename(os.getcwd()) == "notebooks":
    # Cambiamos a la carpeta superior
    os.chdir("..")

# Ensanchamos la salida para que las tablas no se partan
pd.set_option("display.width", 250)

# Dirección pública de los ficheros de este repositorio (rama main)
URL_REPO = "https://raw.githubusercontent.com/dpalacios-data/scouting-laliga-2526/main"
# Fecha a la que se calcula la edad: último día de la temporada 2025/26
FECHA_EDAD = pd.Timestamp("2026-05-31")
# Minutos mínimos para entrar en el cálculo de percentiles (igual que en el TFM)
MINUTOS_MINIMOS = 600
# Grupos que nunca entran en el cálculo de percentiles
GRUPOS_SIN_PERCENTIL = ["sin_definir", "sin_confirmar"]

## 1. Carga

In [3]:
# Mapa de nombres: un jugador del modelo por fila
mapa = pd.read_csv("data/mapa_nombres.csv", dtype={"id_whoscored": "Int64"})
# Acciones de Opta por identificador de WhoScored
eventos = pd.read_csv("data/eventos_por_jugador.csv", dtype={"id_whoscored": "Int64"})
# Posiciones deducidas de las alineaciones
pos_ws = pd.read_csv("data/posiciones_alineaciones.csv")
# Jugadores de mi fichero de posiciones que no jugaron
sin_minutos = pd.read_csv("data/sin_minutos.csv")
# Cruces dudosos que no se aplican
descartados = pd.read_csv("data/cruces_descartados.csv")
# Fichero propio de posiciones
posiciones = pd.read_csv("data/origen/posiciones_laliga_2526.csv")
# Jugadores de Understat
understat = pd.read_csv("data/origen/jugadores_laliga_2526.csv")
# Altura media del bloque defensivo de cada equipo
altura = pd.read_csv("data/origen/altura_defensiva_laliga_2526.csv")
# Clasificación real y por puntos esperados
clasificacion = pd.read_csv("data/origen/clasificacion_xpts_laliga_2526.csv")

## 2. Cuadre de jugadores

In [4]:
# Descartados del cruce posiciones-Understat (salen del fichero de posiciones)
desc_pos = descartados[descartados["cruce"] == "posiciones-understat"]
# Jugadores del fichero de posiciones que entran en el modelo
entran_pos = mapa["nombre_posiciones"].notna().sum()
# Jugadores de campo recuperados de Understat sin posición
recuperados_us = mapa["nombre_posiciones"].isna().sum()
# Filas de Understat que no están en el modelo
fuera_us = understat[~understat["player_id"].isin(mapa["id_understat"])]

# Imprimimos el cuadre línea a línea
print(f"  jugadores en el fichero de posiciones          {len(posiciones):>4}")
print(f"- no jugaron (data/sin_minutos.csv)              {len(sin_minutos):>4}")
print(f"- descartados por cruce dudoso                   {len(desc_pos):>4}")
print(f"= jugadores del fichero de posiciones que entran {len(posiciones) - len(sin_minutos) - len(desc_pos):>4}")
print(f"+ jugadores de campo recuperados de Understat    {recuperados_us:>4}")
print(f"= total de jugadores del modelo                  {len(posiciones) - len(sin_minutos) - len(desc_pos) + recuperados_us:>4}")
print()
print(f"  (aparte) porteros excluidos                    {len(fuera_us):>4}")

# El cuadre tiene que coincidir con las filas del mapa
assert len(posiciones) - len(sin_minutos) - len(desc_pos) == entran_pos, "no cuadra el fichero de posiciones"
# Todo lo que queda fuera de Understat tienen que ser porteros
assert fuera_us["position"].str.contains("GK").all(), "hay jugadores de campo de Understat fuera del modelo"
# Comprobación desde el lado de Understat: cada fila está en el modelo o es un portero excluido
assert len(mapa) + len(fuera_us) == len(understat), "no cuadra Understat"
print(f"\n  comprobación Understat: {len(mapa)} en el modelo + {len(fuera_us)} porteros = {len(understat)} filas")

# Desglose del modelo por origen del cruce
print("\nDesglose por origen del cruce")
print(mapa["origen"].value_counts().to_string())

# Posiciones deducidas de las alineaciones
print("\nPosición deducida de las alineaciones:", len(pos_ws),
      "· confirmada:", int(pos_ws["confirmado"].sum()), "· sin confirmar:", int((~pos_ws["confirmado"]).sum()))

  jugadores en el fichero de posiciones           637
- no jugaron (data/sin_minutos.csv)               112
- descartados por cruce dudoso                      2
= jugadores del fichero de posiciones que entran  523
+ jugadores de campo recuperados de Understat      38
= total de jugadores del modelo                   561

  (aparte) porteros excluidos                      39

  comprobación Understat: 561 en el modelo + 39 porteros = 600 filas

Desglose por origen del cruce
origen
directo                   466
recuperado_36              36
understat_sin_posicion     36
recuperado_15              15
traspasado                  6
recuperado_grafia           2

Posición deducida de las alineaciones: 38 · confirmada: 10 · sin confirmar: 28


## 3. dim_equipo

Correspondencia explícita de nombres de equipo. Los CSV de origen no se tocan.

In [5]:
# Nombres de la altura defensiva y de WhoScored que no coinciden con mi fichero
EQUIPOS = {
    # Abreviatura de WhoScored
    "Atletico": "Atletico Madrid",
    # Nombre oficial completo del Alavés
    "Deportivo Alaves": "Alaves",
}
# Aplicamos la correspondencia a la altura defensiva
altura["equipo"] = altura["equipo"].replace(EQUIPOS)
# Aplicamos la correspondencia a la clasificación
clasificacion["equipo"] = clasificacion["equipo"].replace(EQUIPOS)

# Los 20 equipos, ordenados alfabéticamente
equipos = sorted(posiciones["equipo"].unique())
# Comprobamos que las tres fuentes de equipo coinciden
assert set(altura["equipo"]) == set(equipos) == set(clasificacion["equipo"]), "nombres de equipo distintos"

# Creamos la dimensión con una clave numérica
dim_equipo = pd.DataFrame({"id_equipo": range(1, len(equipos) + 1), "equipo": equipos})
# Nombre para enseñar en el panel: con tildes y como se conoce a cada club ("equipo" sigue siendo el nombre de cruce)
NOMBRE_EQUIPO = {
    "Alaves": "Alavés", "Athletic Club": "Athletic Club", "Atletico Madrid": "Atlético de Madrid",
    "Barcelona": "Barcelona", "Celta Vigo": "Celta", "Elche": "Elche", "Espanyol": "Espanyol", "Getafe": "Getafe",
    "Girona": "Girona", "Levante": "Levante", "Mallorca": "Mallorca", "Osasuna": "Osasuna",
    "Rayo Vallecano": "Rayo Vallecano", "Real Betis": "Real Betis", "Real Madrid": "Real Madrid",
    "Real Oviedo": "Real Oviedo", "Real Sociedad": "Real Sociedad", "Sevilla": "Sevilla", "Valencia": "Valencia",
    "Villarreal": "Villarreal",
}
# Los 20 equipos tienen que tener su nombre para mostrar, ni uno más ni uno menos
assert set(NOMBRE_EQUIPO) == set(equipos), "la lista de nombres para mostrar no cuadra con los equipos"
# Lo ponemos justo detrás del nombre de cruce
dim_equipo.insert(2, "nombre_equipo", dim_equipo["equipo"].map(NOMBRE_EQUIPO))
# Añadimos la altura defensiva
dim_equipo = dim_equipo.merge(altura, on="equipo", how="left")
# Añadimos los puntos reales y los esperados
dim_equipo = dim_equipo.merge(clasificacion[["equipo", "puntos", "xpts"]].rename(
    columns={"xpts": "puntos_esperados"}), on="equipo", how="left")
# Nombre de fichero del escudo: minúsculas, sin acentos y con guiones bajos
dim_equipo["escudo"] = [unidecode(e).lower().replace(" ", "_") + ".png" for e in dim_equipo["equipo"]]
# Colores de la camiseta, calculados en el cuaderno 05 a partir de la ficha de cada club en Wikipedia
colores = pd.read_csv("data/colores_equipos.csv")
# Los añadimos por nombre de equipo
dim_equipo = dim_equipo.merge(colores[["equipo", "color_primario", "color_secundario", "color_origen",
                                       "color_secundario_origen"]], on="equipo", how="left")
# Dirección pública del escudo en este repositorio, para Power BI y Tableau
dim_equipo.insert(dim_equipo.columns.get_loc("escudo") + 1, "url_escudo", URL_REPO + "/images/escudos/" + dim_equipo["escudo"])
# Los 20 equipos tienen que tener color
assert dim_equipo["color_primario"].notna().all(), "falta el color de algún equipo"
# Mostramos la tabla
print(dim_equipo.to_string(index=False))

 id_equipo          equipo      nombre_equipo  altura_defensiva  puntos  puntos_esperados              escudo                                                                                                    url_escudo color_primario color_secundario         color_origen color_secundario_origen
         1          Alaves             Alavés             32.40      43             52.58          alaves.png          https://raw.githubusercontent.com/dpalacios-data/scouting-laliga-2526/main/images/escudos/alaves.png        #0066e1          #ffffff wikipedia_equipacion    wikipedia_equipacion
         2   Athletic Club      Athletic Club             37.73      45             56.90   athletic_club.png   https://raw.githubusercontent.com/dpalacios-data/scouting-laliga-2526/main/images/escudos/athletic_club.png        #ff0b0b          #ffffff wikipedia_equipacion    wikipedia_equipacion
         3 Atletico Madrid Atlético de Madrid             33.29      69             61.83 atletico_madrid.png

## 4. dim_posicion

In [6]:
# Cada posición y su grupo de comparación; pivote y mediocentro van juntos, como en el TFM
POSICIONES = [
    ("central", "central"),
    ("lateral", "lateral"),
    ("pivote", "pivote_mediocentro"),
    ("mediocentro", "pivote_mediocentro"),
    ("mediapunta", "mediapunta"),
    ("extremo", "extremo"),
    ("delantero", "delantero"),
    ("sin_definir", "sin_definir"),
    ("sin_confirmar", "sin_confirmar"),
]
# Creamos la dimensión con una clave numérica
dim_posicion = pd.DataFrame(POSICIONES, columns=["posicion", "grupo"])
# Insertamos la clave al principio
dim_posicion.insert(0, "id_posicion", range(1, len(dim_posicion) + 1))
# Mostramos la tabla
print(dim_posicion.to_string(index=False))

 id_posicion      posicion              grupo
           1       central            central
           2       lateral            lateral
           3        pivote pivote_mediocentro
           4   mediocentro pivote_mediocentro
           5    mediapunta         mediapunta
           6       extremo            extremo
           7     delantero          delantero
           8   sin_definir        sin_definir
           9 sin_confirmar      sin_confirmar


## 5. dim_jugador

In [7]:
# Partimos del mapa
jug = mapa.copy()
# Nombre del jugador en el modelo: el de mi fichero si existe, si no el de Understat
jug["jugador"] = jug["nombre_posiciones"].fillna(jug["nombre_understat"])

# Posición del fichero propio para quien está en él
jug = jug.merge(posiciones.rename(columns={"jugador": "nombre_posiciones"}),
                on=["nombre_posiciones", "equipo"], how="left")
# Origen de la posición para los del fichero propio
jug["posicion_origen"] = np.where(jug["nombre_posiciones"].notna(), "fichero_propio", "alineaciones_whoscored")

# Posiciones de las alineaciones confirmadas por David
confirmadas = pos_ws[pos_ws["confirmado"]].set_index("id_understat")["posicion_propuesta"]
# Los recuperados de Understat toman su posición confirmada o quedan sin confirmar
sin_pos = jug["posicion_origen"] == "alineaciones_whoscored"
# Asignamos la posición
jug.loc[sin_pos, "posicion"] = jug.loc[sin_pos, "id_understat"].map(confirmadas).fillna("sin_confirmar")

# Reparto de partidos con eventos por club (útil sobre todo en los traspasados)
jug = jug.merge(eventos[["id_whoscored", "equipos_eventos"]], on="id_whoscored", how="left")
# Lo llamamos como se acordó
jug = jug.rename(columns={"equipos_eventos": "equipos_temporada", "minutos_us": "minutos", "partidos_us": "partidos"})

# Clave propia: ordenamos por equipo, nombre e identificador de Understat para que sea estable
jug = jug.sort_values(["equipo", "jugador", "id_understat"]).reset_index(drop=True)
# Numeramos desde 1
jug["id_jugador"] = range(1, len(jug) + 1)
# Traemos la clave de equipo
jug = jug.merge(dim_equipo[["id_equipo", "equipo"]], on="equipo", how="left")
# Traemos la clave de posición y su grupo
jug = jug.merge(dim_posicion, on="posicion", how="left")

# Retratos de Transfermarkt emparejados en el cuaderno 06 (solo los aplicados: claros y con foto propia)
fotos = pd.read_csv("data/fotos_transfermarkt.csv")
# Jugadores emparejados con Transfermarkt (tengan retrato propio o no): datos de scouting
emparejados = fotos[fotos["estado"].isin(["claro", "confirmado_autor"])]
# Nos quedamos con los aplicados
fotos = fotos[fotos["aplicado"]]
# Comprobamos que la clave apunta al mismo jugador
assert (jug.set_index("id_jugador").loc[fotos["id_jugador"], "jugador"].values == fotos["jugador"].values).all()
# Url del retrato e identificador de Transfermarkt
# Url original en Transfermarkt del retrato que se recortó (versión grande), del cuaderno 08
recortes = pd.read_csv("data/retratos_recortados.csv")
# Los recortados tienen que ser exactamente los aplicados
assert set(recortes["id_jugador"]) == set(fotos["id_jugador"]), "retratos recortados y aplicados no coinciden"
# La añadimos como url de origen
jug = jug.merge(recortes[["id_jugador", "url_original"]].rename(columns={"url_original": "url_foto_transfermarkt"}),
                on="id_jugador", how="left")
# Datos de scouting e identificador de Transfermarkt de todos los emparejados
jug = jug.merge(emparejados[["id_jugador", "id_transfermarkt", "fecha_nacimiento", "altura_cm", "pie", "nacionalidad",
                             "nacionalidad_2", "valor_mercado_tm", "fecha_descarga"]].rename(
    columns={"fecha_descarga": "fecha_datos_tm"}), on="id_jugador", how="left")
# Fecha de nacimiento como fecha
nacimiento = pd.to_datetime(jug["fecha_nacimiento"])
# Edad en años cumplidos a la fecha de referencia: se resta un año si aún no había cumplido
jug["edad"] = (FECHA_EDAD.year - nacimiento.dt.year
               - ((nacimiento.dt.month > FECHA_EDAD.month)
                  | ((nacimiento.dt.month == FECHA_EDAD.month) & (nacimiento.dt.day > FECHA_EDAD.day))).astype("Int64")).astype("Int64")
# Altura y valor son enteros que pueden faltar
jug[["altura_cm", "valor_mercado_tm"]] = jug[["altura_cm", "valor_mercado_tm"]].astype("Int64")
# El identificador es un entero que puede faltar
jug["id_transfermarkt"] = jug["id_transfermarkt"].astype("Int64")
# Origen del retrato: Transfermarkt o retrato de iniciales de este repositorio (cuaderno 07)
jug["foto_origen"] = np.where(jug["url_foto_transfermarkt"].notna(), "transfermarkt", "iniciales")
# Los que no tienen retrato de Transfermarkt apuntan a su retrato de iniciales, con nombre de fichero por id_jugador
# Todos los retratos, recortados de Transfermarkt (cuaderno 08) o de iniciales (cuaderno 07), están en este
# repositorio con el nombre de fichero por id_jugador
jug["url_foto"] = URL_REPO + "/images/jugadores/" + jug["id_jugador"].astype(str) + ".png"
# Los 561 tienen que tener una url https
assert jug["url_foto"].str.startswith("https://").all()

# Columnas de la dimensión
COLUMNAS_DIM = ["id_jugador", "jugador", "id_equipo", "id_posicion", "minutos", "partidos",
                "posicion_origen", "equipos_temporada", "fecha_nacimiento", "edad", "altura_cm", "pie",
                "nacionalidad", "nacionalidad_2", "valor_mercado_tm", "fecha_datos_tm", "url_foto", "foto_origen", "url_foto_transfermarkt",
                "id_understat", "id_whoscored", "id_transfermarkt"]
# Construimos la dimensión
dim_jugador = jug[COLUMNAS_DIM].copy()
# Mostramos un vistazo
print(dim_jugador.head().to_string(index=False))

 id_jugador        jugador  id_equipo  id_posicion  minutos  partidos posicion_origen equipos_temporada fecha_nacimiento  edad  altura_cm       pie nacionalidad nacionalidad_2  valor_mercado_tm fecha_datos_tm                                                                                          url_foto   foto_origen                                                            url_foto_transfermarkt  id_understat  id_whoscored  id_transfermarkt
          1   Abde Rebbach          1            6     1479        30  fichero_propio         Alaves 30       1998-08-11    27        176   Derecho      Argelia         España           2500000     2026-11-16 https://raw.githubusercontent.com/dpalacios-data/scouting-laliga-2526/main/images/jugadores/1.png transfermarkt https://img.a.transfermarkt.technology/portrait/big/718034-1705337308.jpg?lm=4711         11825        447835            718034
          2    Aitor Mañas          1            7      105         9  fichero_propio          Alaves 

## 6. hechos_jugador: métricas por 90 minutos

- De Understat: goles, xG, asistencias, xA, tiros, pases clave, xGChain y xGBuildup (y sus versiones sin penaltis).
- De Opta, por identificador de WhoScored: entradas intentadas y exitosas, intercepciones, despejes, pases
  bloqueados, duelos aéreos disputados y ganados, centros y toques en el área.
- Dos porcentajes de éxito, que no van por 90: `pct_entradas` y `pct_duelos_aereos`.

Si un jugador no tiene identificador de WhoScored, sus métricas de Opta quedan **vacías**, nunca a cero:
`tiene_eventos` distingue "no hizo la acción" (0) de "no tengo el dato" (vacío).

In [8]:
# Métricas de Understat y su nombre en el modelo
UNDERSTAT = {"goals": "goles", "np_goals": "goles_sin_penalti", "xg": "xg", "np_xg": "xg_sin_penalti",
             "assists": "asistencias", "xa": "xa", "shots": "tiros", "key_passes": "pases_clave",
             "xg_chain": "xg_chain", "xg_buildup": "xg_buildup"}
# Métricas de Opta (ya con su nombre en el modelo)
OPTA = ["entradas_intentadas", "entradas_exitosas", "intercepciones", "despejes", "pases_bloqueados",
        "duelos_aereos_disputados", "duelos_aereos_ganados", "centros", "toques_area"]
# Porcentajes de éxito: (columna, aciertos, intentos)
PORCENTAJES = [("pct_entradas", "entradas_exitosas", "entradas_intentadas"),
               ("pct_duelos_aereos", "duelos_aereos_ganados", "duelos_aereos_disputados")]

# Totales de Understat de cada jugador
hechos = jug[["id_jugador", "id_understat", "id_whoscored", "minutos", "grupo"]].merge(
    understat[["player_id"] + list(UNDERSTAT)].rename(columns={"player_id": "id_understat", **UNDERSTAT}),
    on="id_understat", how="left", validate="one_to_one")
# Totales de Opta; quien no tiene identificador se queda sin fila y sus valores quedan vacíos
hechos = hechos.merge(eventos[["id_whoscored"] + OPTA], on="id_whoscored", how="left")
# Tiene eventos quien tiene identificador de WhoScored
hechos["tiene_eventos"] = hechos["id_whoscored"].notna()

# Métricas que se pasan a valores por 90 minutos
POR90 = list(UNDERSTAT.values()) + OPTA
# Calculamos cada métrica por 90 minutos; un vacío sigue vacío
for m in POR90:
    # Total dividido entre minutos y multiplicado por 90
    hechos[m + "_90"] = hechos[m] / hechos["minutos"] * 90

# Calculamos los porcentajes de éxito; sin intentos, el porcentaje queda vacío (no es un 0 %)
for col, aciertos, intentos in PORCENTAJES:
    # Aciertos entre intentos, en tanto por cien
    hechos[col] = hechos[aciertos] / hechos[intentos].where(hechos[intentos] > 0) * 100

# Columnas de métrica del modelo: por 90 y porcentajes
COLUMNAS_METRICA = [m + "_90" for m in POR90] + [c for c, _, _ in PORCENTAJES]
# Columnas que salen de Opta
COLUMNAS_OPTA = [m + "_90" for m in OPTA] + [c for c, _, _ in PORCENTAJES]

# Control: los jugadores sin eventos tienen las métricas de Opta vacías
assert hechos.loc[~hechos["tiene_eventos"], COLUMNAS_OPTA].isna().all().all()
# Los que tienen eventos no tienen huecos en las métricas por 90
assert hechos.loc[hechos["tiene_eventos"], [m + "_90" for m in OPTA]].notna().all().all()
# Jugadores con eventos pero sin intentos, cuyo porcentaje queda vacío
for col, _, intentos in PORCENTAJES:
    # Recuento por porcentaje
    print(f"{col}: {int((hechos['tiene_eventos'] & hechos[col].isna()).sum())} jugadores con eventos y 0 {intentos}")

pct_entradas: 35 jugadores con eventos y 0 entradas_intentadas
pct_duelos_aereos: 39 jugadores con eventos y 0 duelos_aereos_disputados


## 7. Percentiles por grupo de posición

- **Grupo de referencia:** jugadores de su mismo grupo de posición con 600 minutos o más.
- **Fórmula:** porcentaje de jugadores del grupo de referencia con un valor **estrictamente menor**. Es la
  fórmula de las cifras publicadas en el TFM (Valjent p9, Llorente p98). El mejor del grupo no llega a 100.
- La misma fórmula se aplica a todos los jugadores del grupo, también a los de menos de 600 minutos,
  que se comparan contra el grupo de referencia pero llevan `percentil_fiable` en falso.
- `sin_definir` y `sin_confirmar` no tienen percentil.

In [9]:
# Un jugador es de referencia si tiene 600 minutos y un grupo con percentil
hechos["referencia"] = (hechos["minutos"] >= MINUTOS_MINIMOS) & ~hechos["grupo"].isin(GRUPOS_SIN_PERCENTIL)


# Calcula el percentil de una métrica dentro de cada grupo de posición
def percentil(tabla, columna):
    # Empezamos con todo vacío
    salida = pd.Series(np.nan, index=tabla.index)
    # Recorremos los grupos que sí llevan percentil
    for grupo in tabla.loc[~tabla["grupo"].isin(GRUPOS_SIN_PERCENTIL), "grupo"].unique():
        # Filas del grupo con dato en esta métrica
        en_grupo = (tabla["grupo"] == grupo) & tabla[columna].notna()
        # Filas de referencia del grupo
        ref = en_grupo & tabla["referencia"]
        # Valores de referencia ordenados
        orden = np.sort(tabla.loc[ref, columna].to_numpy())
        # Porcentaje de jugadores de referencia con un valor estrictamente menor
        salida[en_grupo] = np.searchsorted(orden, tabla.loc[en_grupo, columna].to_numpy(), side="left") / len(orden) * 100
    # Devolvemos la serie
    return salida


# Calculamos el percentil de cada métrica
for c in COLUMNAS_METRICA:
    # Columna con el percentil: pctl_ y el nombre de la métrica sin _90
    hechos["pctl_" + c.removesuffix("_90")] = percentil(hechos, c)

# Columnas de percentil
COLUMNAS_PCTL = ["pctl_" + c.removesuffix("_90") for c in COLUMNAS_METRICA]

# El percentil es fiable solo para los jugadores de referencia
hechos["percentil_fiable"] = hechos["referencia"]

# Jugadores de referencia de cada grupo de posición
n_ref = hechos[hechos["referencia"]].groupby("grupo").size()
# Cada jugador lleva el tamaño del grupo contra el que se calcula su percentil (vacío si no tiene percentil)
hechos["n_grupo_referencia"] = hechos["grupo"].map(n_ref).astype("Int64")
# Control: en todas las métricas el grupo de referencia es el mismo (nadie con 600 minutos tiene huecos)
for m in COLUMNAS_METRICA:
    # Jugadores de referencia con dato en esta métrica
    con_dato = hechos[hechos["referencia"] & hechos[m].notna()].groupby("grupo").size()
    # Tiene que coincidir con el total del grupo
    assert con_dato.equals(n_ref), f"grupo de referencia distinto en {m}"
# Umbral a partir del cual el panel avisa de que la muestra es pequeña
MUESTRA_REDUCIDA = 30
# Aviso para los grupos con 30 jugadores de referencia o menos
hechos["muestra_reducida"] = hechos["n_grupo_referencia"].le(MUESTRA_REDUCIDA).fillna(False).astype(bool)

# Columnas de la tabla de hechos
COLUMNAS_HECHOS = (["id_jugador", "tiene_eventos"] + COLUMNAS_METRICA
                   + COLUMNAS_PCTL + ["percentil_fiable", "n_grupo_referencia", "muestra_reducida"])
# Construimos la tabla de hechos
hechos_jugador = hechos[COLUMNAS_HECHOS].copy()
# Redondeamos las métricas a 4 decimales
hechos_jugador[COLUMNAS_METRICA] = hechos_jugador[COLUMNAS_METRICA].round(4)
# Redondeamos los percentiles a 2 decimales
hechos_jugador[COLUMNAS_PCTL] = hechos_jugador[COLUMNAS_PCTL].round(2)

## 8. Formato largo para la página Ficha

En la Ficha las métricas van en el eje vertical, una fila por métrica, así que hacen falta en formato largo:

- `dim_metrica`: una fila por métrica, con su nombre para el panel, su bloque y su unidad.
- `hechos_metrica`: una fila por jugador y métrica, con el valor y el percentil.
- `hechos_ficha`: los datos descriptivos del jugador como pares atributo-valor, ya formateados como texto.

`hechos_jugador` no cambia: es lo que alimenta la página Buscador.

In [10]:
# (columna de hechos_jugador, nombre, nombre corto, bloque, unidad); el orden dentro del bloque es el de la lista
METRICAS_FICHA = [
    ("intercepciones_90", "Intercepciones", "Intercep.", "defensa", "por_90"),
    ("entradas_exitosas_90", "Entradas exitosas", "Entradas", "defensa", "por_90"),
    ("pct_entradas", "% de entradas exitosas", "% entradas", "defensa", "porcentaje"),
    ("despejes_90", "Despejes", "Despejes", "defensa", "por_90"),
    ("pases_bloqueados_90", "Pases bloqueados", "Bloqueos", "defensa", "por_90"),
    ("duelos_aereos_ganados_90", "Duelos aéreos ganados", "Aéreos", "defensa", "por_90"),
    ("pct_duelos_aereos", "% de duelos aéreos ganados", "% aéreos", "defensa", "porcentaje"),
    ("entradas_intentadas_90", "Entradas intentadas", "Ent. intent.", "defensa", "por_90"),
    ("duelos_aereos_disputados_90", "Duelos aéreos disputados", "Aéreos disp.", "defensa", "por_90"),
    ("pases_clave_90", "Pases clave", "Pases clave", "con_balon", "por_90"),
    ("xg_chain_90", "xG chain", "xGChain", "con_balon", "por_90"),
    ("xg_buildup_90", "xG buildup", "xGBuildup", "con_balon", "por_90"),
    ("centros_90", "Centros", "Centros", "con_balon", "por_90"),
    ("toques_area_90", "Toques en el área", "Toques área", "con_balon", "por_90"),
    ("tiros_90", "Tiros", "Tiros", "con_balon", "por_90"),
    ("goles_90", "Goles", "Goles", "ataque", "por_90"),
    ("goles_sin_penalti_90", "Goles sin penalti", "Goles s/p", "ataque", "por_90"),
    ("xg_90", "xG", "xG", "ataque", "por_90"),
    ("xg_sin_penalti_90", "xG sin penalti", "npxG", "ataque", "por_90"),
    ("asistencias_90", "Asistencias", "Asist.", "ataque", "por_90"),
    ("xa_90", "xA", "xA", "ataque", "por_90"),
]
# Todas las métricas del modelo tienen que estar, ni una más ni una menos
assert sorted(m for m, *_ in METRICAS_FICHA) == sorted(COLUMNAS_METRICA), "dim_metrica no cubre todas las métricas"

# Dimensión de métricas
dim_metrica = pd.DataFrame(METRICAS_FICHA, columns=["metrica", "nombre", "nombre_corto", "bloque", "unidad"])
# Orden dentro de su bloque, empezando en 1
dim_metrica["orden"] = dim_metrica.groupby("bloque").cumcount() + 1
# Clave propia
dim_metrica.insert(0, "id_metrica", range(1, len(dim_metrica) + 1))
# Columnas en orden
dim_metrica = dim_metrica[["id_metrica", "metrica", "nombre", "nombre_corto", "bloque", "orden", "unidad"]]

# Hechos en largo: una fila por jugador y métrica
largo = []
for f in dim_metrica.itertuples():
    # Columna del percentil de esa métrica en hechos_jugador
    pctl = "pctl_" + f.metrica.removesuffix("_90")
    # Valor y percentil de los 561 jugadores; un vacío sigue vacío
    parte = hechos_jugador[["id_jugador", f.metrica, pctl]].rename(columns={f.metrica: "valor", pctl: "percentil"})
    # Clave de la métrica
    parte.insert(1, "id_metrica", f.id_metrica)
    largo.append(parte)
# Juntamos
hechos_metrica = pd.concat(largo, ignore_index=True)
# Percentil sin decimales, entero que puede faltar (misma fórmula; solo se redondea)
hechos_metrica["percentil"] = hechos_metrica["percentil"].round(0).astype("Int64")
# Cuadre: 561 jugadores por cada métrica
assert len(hechos_metrica) == len(dim_jugador) * len(dim_metrica)


# Formatos de texto para la ficha (vacío si no hay dato)
def miles(n):
    # 3330 -> "3.330"
    return f"{int(n):,}".replace(",", ".")


def valor_mercado(v):
    # 4500000 -> "4,5 M€"; 500000 -> "500 mil €"
    if v >= 1_000_000:
        return f"{v / 1_000_000:.1f}".rstrip("0").rstrip(".").replace(".", ",") + " M€"
    return f"{v / 1000:.0f} mil €"


# Nombre de cada posición tal como se lee en el panel
NOMBRE_POSICION = {"sin_definir": "Sin definir", "sin_confirmar": "Posición no confirmada"}
# Datos del jugador con equipo y posición
ficha = jug.rename(columns={"posicion": "pos"})[["id_jugador", "equipo", "pos", "edad", "altura_cm", "pie", "nacionalidad",
                                                 "nacionalidad_2", "minutos", "partidos", "valor_mercado_tm"]]
# Atributos en el orden de la ficha: (atributo, función que da el texto o None)
ATRIBUTOS = [
    ("Equipo", lambda f: NOMBRE_EQUIPO[f.equipo]),
    ("Puesto", lambda f: NOMBRE_POSICION.get(f.pos, f.pos.capitalize())),
    ("Edad", lambda f: f"{int(f.edad)} años" if pd.notna(f.edad) else None),
    ("Altura", lambda f: f"{int(f.altura_cm) / 100:.2f} m".replace(".", ",") if pd.notna(f.altura_cm) else None),
    ("Pie", lambda f: f.pie if pd.notna(f.pie) else None),
    ("Nacionalidad", lambda f: (f.nacionalidad + (f" / {f.nacionalidad_2}" if pd.notna(f.nacionalidad_2) else ""))
     if pd.notna(f.nacionalidad) else None),
    ("Minutos", lambda f: miles(f.minutos)),
    ("Partidos", lambda f: str(int(f.partidos))),
    ("Valor estimado", lambda f: valor_mercado(f.valor_mercado_tm) if pd.notna(f.valor_mercado_tm) else None),
]
# Una fila por jugador y atributo
hechos_ficha = pd.DataFrame([{"id_jugador": f.id_jugador, "orden": n, "atributo": a, "valor": fn(f)}
                             for f in ficha.itertuples() for n, (a, fn) in enumerate(ATRIBUTOS, start=1)])
# Cuadre: 561 jugadores por cada atributo
assert len(hechos_ficha) == len(dim_jugador) * len(ATRIBUTOS)
# Resumen
print(f"dim_metrica: {len(dim_metrica)} métricas · {dim_metrica['bloque'].value_counts().to_dict()}")
print(f"hechos_metrica: {len(hechos_metrica)} filas = {len(dim_jugador)} jugadores x {len(dim_metrica)} métricas"
      f" · valores vacíos: {int(hechos_metrica['valor'].isna().sum())}")
print(f"hechos_ficha: {len(hechos_ficha)} filas = {len(dim_jugador)} jugadores x {len(ATRIBUTOS)} atributos"
      f" · valores vacíos: {hechos_ficha.groupby('atributo')['valor'].apply(lambda s: int(s.isna().sum())).to_dict()}")

dim_metrica: 21 métricas · {'defensa': 9, 'con_balon': 6, 'ataque': 6}
hechos_metrica: 11781 filas = 561 jugadores x 21 métricas · valores vacíos: 129
hechos_ficha: 5049 filas = 561 jugadores x 9 atributos · valores vacíos: {'Altura': 8, 'Edad': 4, 'Equipo': 0, 'Minutos': 0, 'Nacionalidad': 4, 'Partidos': 0, 'Pie': 12, 'Puesto': 0, 'Valor estimado': 9}


## 9. Colores de jugador para la página Comparación

En la página Comparación cada jugador se pinta con un color sacado del de su equipo.

- `dim_jugador.color_jugador`: el `color_primario` de su equipo, aclarado lo justo para que contraste al menos
  3:1 con el fondo del panel (#07090E). Es el color base del club.
- `dim_equipo.color_grado_1`, `color_grado_2` y `color_grado_3`: tres grados muy separados del color del club.
  En Power BI, una medida da a cada jugador seleccionado del mismo club un grado distinto (el de más minutos, el 1).

Un color fijo por jugador no puede distinguir a los 22-33 jugadores de un club con un solo tono: la peor
distancia entre dos de ellos queda por debajo de lo que el ojo distingue. Como en la página se comparan 2 o 3,
basta con 3 grados bien separados entre los seleccionados.

Los colores claros (blanco del Real Madrid, amarillo del Villarreal, celeste del Celta) no se pueden aclarar
más: sus grados se oscurecen, hasta el límite del contraste 3:1.

In [11]:
# Fondo del panel
FONDO_PANEL = "#07090E"


def a_rgb(color):
    # "#rrggbb" -> [r, g, b] entre 0 y 1
    return [int(color.lstrip("#")[i:i + 2], 16) / 255 for i in (0, 2, 4)]


def a_hex(c):
    # [r, g, b] entre 0 y 1 -> "#rrggbb"
    return "#" + "".join(f"{round(min(1, max(0, v)) * 255):02x}" for v in c)


def lineal(v):
    # Componente sRGB a luz lineal
    return v / 12.92 if v <= 0.04045 else ((v + 0.055) / 1.055) ** 2.4


def luminancia(c):
    # Luminancia relativa de la WCAG
    return 0.2126 * lineal(c[0]) + 0.7152 * lineal(c[1]) + 0.0722 * lineal(c[2])


def contraste(a, b):
    # Razón de contraste de la WCAG entre dos colores
    claro, oscuro = sorted((luminancia(a), luminancia(b)), reverse=True)
    # Fórmula de la WCAG
    return (claro + 0.05) / (oscuro + 0.05)


def aclara(c, t):
    # Mezcla con blanco en proporción t
    return [v + (1 - v) * t for v in c]


def oscurece(c, t):
    # Mezcla con negro en proporción t
    return [v * (1 - t) for v in c]


def lab(c):
    # sRGB -> CIELAB (iluminante D65)
    r, g, b = (lineal(v) for v in c)
    # A XYZ normalizado por el blanco de referencia
    x, y, z = ((0.4124 * r + 0.3576 * g + 0.1805 * b) / 0.95047, 0.2126 * r + 0.7152 * g + 0.0722 * b,
               (0.0193 * r + 0.1192 * g + 0.9505 * b) / 1.08883)
    # Función de compresión de CIELAB
    f = lambda t: t ** (1 / 3) if t > 216 / 24389 else (24389 / 27 * t + 16) / 116
    # L, a, b
    return 116 * f(y) - 16, 500 * (f(x) - f(y)), 200 * (f(y) - f(z))


def delta_e2000(c1, c2):
    # Distancia de color CIEDE2000: por debajo de ~2 el ojo apenas distingue; por encima de 10, a simple vista
    (L1, a1, b1), (L2, a2, b2) = lab(c1), lab(c2)
    # Croma medio y corrección G del eje a
    cm = (np.hypot(a1, b1) + np.hypot(a2, b2)) / 2
    g = 0.5 * (1 - np.sqrt(cm ** 7 / (cm ** 7 + 25 ** 7)))
    # Croma y tono corregidos
    a1, a2 = a1 * (1 + g), a2 * (1 + g)
    c1p, c2p = np.hypot(a1, b1), np.hypot(a2, b2)
    h1, h2 = np.degrees(np.arctan2(b1, a1)) % 360, np.degrees(np.arctan2(b2, a2)) % 360
    # Diferencia de tono, por el camino corto
    dh = 0 if c1p * c2p == 0 else (h2 - h1 if abs(h2 - h1) <= 180 else h2 - h1 - 360 if h2 > h1 else h2 - h1 + 360)
    dH = 2 * np.sqrt(c1p * c2p) * np.sin(np.radians(dh / 2))
    # Medias de luminosidad, croma y tono
    lm, cmp_ = (L1 + L2) / 2, (c1p + c2p) / 2
    hm = h1 + h2 if c1p * c2p == 0 else ((h1 + h2) / 2 if abs(h1 - h2) <= 180 else (h1 + h2 + 360) / 2 if h1 + h2 < 360 else (h1 + h2 - 360) / 2)
    # Pesos de la fórmula
    t = (1 - 0.17 * np.cos(np.radians(hm - 30)) + 0.24 * np.cos(np.radians(2 * hm))
         + 0.32 * np.cos(np.radians(3 * hm + 6)) - 0.20 * np.cos(np.radians(4 * hm - 63)))
    sl = 1 + 0.015 * (lm - 50) ** 2 / np.sqrt(20 + (lm - 50) ** 2)
    sc, sh = 1 + 0.045 * cmp_, 1 + 0.015 * cmp_ * t
    rt = -2 * np.sqrt(cmp_ ** 7 / (cmp_ ** 7 + 25 ** 7)) * np.sin(np.radians(60 * np.exp(-((hm - 275) / 25) ** 2)))
    # Distancia
    return float(np.sqrt(((L2 - L1) / sl) ** 2 + ((c2p - c1p) / sc) ** 2 + (dH / sh) ** 2 + rt * ((c2p - c1p) / sc) * (dH / sh)))


# Mezcla máxima con blanco para el grado más claro: no llega al blanco puro
ACLARADO_MAXIMO = 0.85
# Por encima de esta luminancia el color es claro y sus grados se oscurecen
LUMINANCIA_CLARO = 0.5
# Grados de cada equipo
grados = {}
for eq in dim_equipo.itertuples():
    base = a_rgb(eq.color_primario)
    if luminancia(base) > LUMINANCIA_CLARO:
        # Color claro: ya contrasta; se oscurece hasta justo antes de perder el 3:1
        t_max = 0.0
        # Mayor mezcla con negro que sigue contrastando 3:1
        while contraste(oscurece(base, t_max + 0.01), a_rgb(FONDO_PANEL)) >= 3:
            t_max += 0.01
        colores = [oscurece(base, t_max * g) for g in (0, 0.5, 1)]
    else:
        # Color oscuro o medio: se aclara lo justo para el 3:1 y desde ahí hasta el máximo
        t_min = 0.0
        # Primera mezcla con blanco que contrasta 3:1
        while contraste(aclara(base, t_min), a_rgb(FONDO_PANEL)) < 3:
            t_min += 0.005
        colores = [aclara(base, t_min + (ACLARADO_MAXIMO - t_min) * g) for g in (0, 0.5, 1)]
    # Los tres grados contrastan con el fondo
    assert all(contraste(c, a_rgb(FONDO_PANEL)) >= 3 for c in colores), eq.equipo
    grados[eq.id_equipo] = colores

# Columnas de grado en dim_equipo
for k in range(3):
    dim_equipo[f"color_grado_{k + 1}"] = dim_equipo["id_equipo"].map(lambda i: a_hex(grados[i][k]))
# Color base del jugador: el grado 1 de su equipo
dim_jugador["color_jugador"] = dim_jugador["id_equipo"].map(dim_equipo.set_index("id_equipo")["color_grado_1"])

# Peor caso: en cada equipo, la menor distancia entre dos de sus grados
peor = pd.DataFrame([{"equipo": eq.nombre_equipo,
                      "grados": " ".join(a_hex(c) for c in grados[eq.id_equipo]),
                      "distancia_minima": round(min(delta_e2000(grados[eq.id_equipo][a], grados[eq.id_equipo][b])
                                                    for a, b in [(0, 1), (0, 2), (1, 2)]), 1)}
                     for eq in dim_equipo.itertuples()]).sort_values("distancia_minima")
print(peor.to_string(index=False))

            equipo                  grados  distancia_minima
             Celta #a3d0f4 #7798b2 #4b6070              15.7
Atlético de Madrid #ff0000 #ff6c6c #ffd9d9              16.2
          Mallorca #ff0000 #ff6c6c #ffd9d9              16.2
            Girona #ff0000 #ff6c6c #ffd9d9              16.2
    Rayo Vallecano #ff0000 #ff6c6c #ffd9d9              16.2
           Osasuna #ff0000 #ff6c6c #ffd9d9              16.2
        Real Betis #349b5c #8ac5a1 #e1f0e7              16.5
     Athletic Club #ff0b0b #ff7373 #ffdada              16.7
          Valencia #e33e07 #ef9070 #fbe2da              18.5
       Real Madrid #ffffff #afafaf #5e5e5e              18.7
           Sevilla #e20000 #ee6c6c #fbd9d9              19.0
        Villarreal #ffff00 #b0b000 #616100              19.0
           Levante #894a56 #b9939a #e8dcde              19.8
     Real Sociedad #475cad #95a1d0 #e3e7f3              20.6
            Alavés #0066e1 #6ca7ee #d9e8fa              21.0
             Elche #0081

## 10. Guardado

In [12]:
# Carpeta del modelo
os.makedirs("data/modelo", exist_ok=True)
# Guardamos cada tabla
for nombre, tabla in [("dim_equipo", dim_equipo), ("dim_posicion", dim_posicion),
                      ("dim_jugador", dim_jugador), ("hechos_jugador", hechos_jugador),
                      ("dim_metrica", dim_metrica), ("hechos_metrica", hechos_metrica), ("hechos_ficha", hechos_ficha)]:
    # Escribimos el CSV sin índice
    tabla.to_csv(f"data/modelo/{nombre}.csv", index=False)
    # Informamos de filas y columnas
    print(f"{nombre}.csv: {tabla.shape[0]} filas, {tabla.shape[1]} columnas")

dim_equipo.csv: 20 filas, 15 columnas
dim_posicion.csv: 9 filas, 3 columnas
dim_jugador.csv: 561 filas, 23 columnas
hechos_jugador.csv: 561 filas, 47 columnas
dim_metrica.csv: 21 filas, 7 columnas
hechos_metrica.csv: 11781 filas, 4 columnas
hechos_ficha.csv: 5049 filas, 4 columnas


## 11. Comprobaciones de calidad

In [13]:
# Releemos las tablas guardadas, que son las que verá Power BI
d_eq = pd.read_csv("data/modelo/dim_equipo.csv")
d_pos = pd.read_csv("data/modelo/dim_posicion.csv")
d_jug = pd.read_csv("data/modelo/dim_jugador.csv")
h = pd.read_csv("data/modelo/hechos_jugador.csv")

# 1. Jugadores en total y con percentil fiable
print("1. Jugadores en total:", len(d_jug), "· con percentil fiable:", int(h["percentil_fiable"].sum()))

# 2. Claves de jugador repetidas
print("2. id_jugador repetidos en dim_jugador:", int(d_jug["id_jugador"].duplicated().sum()),
      "· en hechos_jugador:", int(h["id_jugador"].duplicated().sum()))

# 3. Integridad referencial
print("3. Claves huérfanas:")
print("   hechos.id_jugador fuera de dim_jugador:", int((~h["id_jugador"].isin(d_jug["id_jugador"])).sum()))
print("   dim_jugador.id_equipo fuera de dim_equipo:", int((~d_jug["id_equipo"].isin(d_eq["id_equipo"])).sum()))
print("   dim_jugador.id_posicion fuera de dim_posicion:", int((~d_jug["id_posicion"].isin(d_pos["id_posicion"])).sum()))
print("   jugadores de dim_jugador sin fila de hechos:", int((~d_jug["id_jugador"].isin(h["id_jugador"])).sum()))

# 4. Jugadores por grupo de posición y cuántos pasan de 600 minutos
reparto = d_jug.merge(d_pos, on="id_posicion").assign(con_600=lambda t: t["minutos"] >= MINUTOS_MINIMOS)
# Tabla por grupo
por_grupo = reparto.groupby("grupo").agg(jugadores=("id_jugador", "size"), con_600_min=("con_600", "sum"))
print("\n4. Jugadores por grupo de posición")
print(por_grupo.sort_values("con_600_min", ascending=False).to_string())
# Grupos con percentil y muestra de referencia corta
cortos = por_grupo[(por_grupo["con_600_min"] <= 30) & ~por_grupo.index.isin(GRUPOS_SIN_PERCENTIL)]
print("   grupos con percentil y 30 o menos jugadores de referencia:", list(cortos.index) or "ninguno")

# 5. Valores ausentes por métrica
ausentes = h[[c for c in h.columns if c.endswith("_90") or c.startswith("pct_")]].isna().sum()
print("\n5. Métricas con valores ausentes (jugadores)")
print(ausentes[ausentes > 0].to_string() if (ausentes > 0).any() else "   ninguna")

1. Jugadores en total: 561 · con percentil fiable: 368
2. id_jugador repetidos en dim_jugador: 0 · en hechos_jugador: 0
3. Claves huérfanas:
   hechos.id_jugador fuera de dim_jugador: 0
   dim_jugador.id_equipo fuera de dim_equipo: 0
   dim_jugador.id_posicion fuera de dim_posicion: 0
   jugadores de dim_jugador sin fila de hechos: 0

4. Jugadores por grupo de posición
                    jugadores  con_600_min
grupo                                     
pivote_mediocentro        108           87
lateral                    94           73
central                    95           71
extremo                    86           59
delantero                  83           50
mediapunta                 39           28
sin_definir                28            5
sin_confirmar              28            0
   grupos con percentil y 30 o menos jugadores de referencia: ['mediapunta']

5. Métricas con valores ausentes (jugadores)
entradas_intentadas_90          5
entradas_exitosas_90            5
interce